# Timefolio Engine — Quickstart

This notebook walks through the most common use-cases of the `timefolio` library.

**Prerequisites**

```bash
# Install the library from the project root
pip install -e .            # core (requests only)
```

**Credentials**

Create a `.env` file in the project root (never commit this file):

```
TIMEFOLIO_EMAIL=you@example.com
TIMEFOLIO_PASSWORD=your_password
TIMEFOLIO_PF_ID=18762
```

Or set the environment variables directly before running this notebook.

## 0. Setup

In [1]:
import logging
import os

# Optional: load credentials from a .env file in the project root.
# Remove this block if you prefer to set env vars another way.
try:
    from dotenv import load_dotenv
    load_dotenv()  # reads ../.env relative to examples/
    load_dotenv(dotenv_path="../.env", override=False)
    
except ImportError:
    pass  # python-dotenv is optional; credentials can come from the shell env

# Show INFO-level log messages from the library so you can follow what's happening.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)

In [2]:
from timefolio import TimefolioAPIClient
from timefolio import TimefolioTrader

## 1. Authentication

`TimefolioAPIClient` manages a single `requests.Session`.  
Call `login()` once — the Bearer token is automatically attached to every
subsequent request.

In [3]:
EMAIL    = os.environ["TIMEFOLIO_EMAIL"]
PASSWORD = os.environ["TIMEFOLIO_PASSWORD"]
PF_ID    = os.environ["TIMEFOLIO_PF_ID"]

api = TimefolioAPIClient(email=EMAIL, password=PASSWORD)

if not api.login():
    raise RuntimeError("Login failed — check your credentials in .env")

print(f"Logged in.  Active portfolio ID: {PF_ID}")

2026-04-25 04:30:53,227 [INFO] timefolio.api_client: Attempting to log in as cmschs03@naver.com ...
2026-04-25 04:30:53,520 [INFO] timefolio.api_client: Login successful.


Logged in.  Active portfolio ID: 19252


## 2. Creating a Trader

`TimefolioTrader` wraps the authenticated client with trading-specific methods.

In [4]:
trader = TimefolioTrader(api_client=api, pf_id=PF_ID)
print(f"Trader ready (pfId={trader.pf_id})")

Trader ready (pfId=19252)


## 3. Switch Tournament (optional)

If you are enrolled in multiple contests, use `set_tournament()` to switch
the active portfolio by matching a substring of the contest name.

In [ ]:
# Example — change "연습용 대회" to the actual contest name you want to trade in.
# trader.set_tournament(target_name="연습용 대회")
# print(f"Now using pfId={trader.pf_id}")

## 4. 실시간 연결 & 포트폴리오 조회

`connect_realtime()`은 SignalR WebSocket으로 서버에 연결해 보유잔고·주문·포지션 데이터를 실시간으로 수신합니다.  
주문 취소 헬퍼(`cancel_all_pending`, `cancel_pending_by_stock`)도 이 데이터를 사용합니다.

| 메서드 | 설명 |
|---|---|
| `connect_realtime()` | SignalR 연결 시작 |
| `get_holdings()` | 보유잔고 (NAV, 평가금액 등) |
| `get_orders()` | 전체 주문 목록 |
| `get_positions()` | 보유 포지션 목록 |
| `get_pending_orders()` | 미접수 주문만 필터 (`state == "Generated"`) |
| `get_error_orders()` | 에러 주문만 필터 |
| `disconnect_realtime()` | SignalR 연결 종료 |

In [ ]:
import time

trader.connect_realtime()
time.sleep(3)  # 데이터 수신 대기

print("보유잔고 :", trader.get_holdings())
print("주문목록 :", trader.get_orders())

print("포지션   :", trader.get_positions())
print("미접수   :", trader.get_pending_orders())
print("에러주문 :", trader.get_error_orders())

trader.disconnect_realtime()

2026-04-25 03:27:27,489 [INFO] timefolio.trader: SignalR 연결 완료
2026-04-25 03:27:27,497 [INFO] SignalRCoreClient: on_open not defined
2026-04-25 03:27:27,549 [INFO] timefolio.trader: 보유잔고 업데이트: {'Id': 945260, 'd': '2026-04-24', 'pfId': 19252, 'prft': 14386160, 'na': 1263870465, 'rt': 1.01151368, 'nav': 1263.8704, 'stkAmt': 1240691925, 'buyAmt': 0, 'sellAmt': 0, 'pfStd': 40.9499, 'active': True, 'rank': 122, 'stkViolCnt': 0, 'stkViolExc': 0, 'smcapViolExc': 0, 'sectViolCnt': 0, 'sectViolExc': 0, 'nPeers': 0, 'ctstNm': None, 'userNm': None, 'userNick': None, 'violCnt': 0, 'violExc': 0}
2026-04-25 03:27:27,550 [INFO] timefolio.trader: 주문 업데이트: []
2026-04-25 03:27:27,550 [INFO] timefolio.trader: 포지션 업데이트: [{'Id': 4320347, 'd': '2026-04-24', 'pfId': 19252, 'prodId': 'A000270', 'prodNm': '기아', 'rstr': 'None', 'posYd': 33, 'prcYd': 158400, 'bQty': 0, 'bPrc': 0, 'sQty': 0, 'sPrc': 0, 'close': 153400, 'div': 0, 'fee': 0, 'tax': 0, 'pos': 33, 'prft': -165000, 'prftKrw': None, 'avgPrc': 152855.727

보유잔고 : {'Id': 945260, 'd': '2026-04-24', 'pfId': 19252, 'prft': 14386160, 'na': 1263870465, 'rt': 1.01151368, 'nav': 1263.8704, 'stkAmt': 1240691925, 'buyAmt': 0, 'sellAmt': 0, 'pfStd': 40.9499, 'active': True, 'rank': 122, 'stkViolCnt': 0, 'stkViolExc': 0, 'smcapViolExc': 0, 'sectViolCnt': 0, 'sectViolExc': 0, 'nPeers': 0, 'ctstNm': None, 'userNm': None, 'userNick': None, 'violCnt': 0, 'violExc': 0}
주문목록 : []
포지션   : [{'Id': 4320347, 'd': '2026-04-24', 'pfId': 19252, 'prodId': 'A000270', 'prodNm': '기아', 'rstr': 'None', 'posYd': 33, 'prcYd': 158400, 'bQty': 0, 'bPrc': 0, 'sQty': 0, 'sPrc': 0, 'close': 153400, 'div': 0, 'fee': 0, 'tax': 0, 'pos': 33, 'prft': -165000, 'prftKrw': None, 'avgPrc': 152855.7273, 'chPct': None, 'wei': 0.4, 'yWei': 0.41, 'ctstNm': None, 'userNm': None, 'userNick': None, 'sec': '25'}, {'Id': 4320348, 'd': '2026-04-24', 'pfId': 19252, 'prodId': 'A000660', 'prodNm': 'SK하이닉스', 'rstr': 'None', 'posYd': 103, 'prcYd': 1225000, 'bQty': 0, 'bPrc': 0, 'sQty': 0, 'sPrc': 

## 5. Market Order (Immediate Execution)

Omit `hm0` / `hm1` to execute at the current market price right now.

| Parameter | Description |
|---|---|
| `prod_id` | Ticker with `A` prefix (e.g. `A005930` = Samsung Electronics) |
| `weight` | Fraction of total NAV — `0.05` = 5 % |
| `ls` | `"L"` Buy / `"S"` Sell |
| `limit_idx` | Order-book aggressiveness 1–10 (default 5) |

In [ ]:
# Buy Samsung Electronics (A005930) at 5 % portfolio weight, execute immediately.
result = trader.order(
    prod_id="A005930",
    weight=0.05,
    ls="L",
)
print(result)

## 6. Scheduled / TWAP Order

Set `hm0` (start time) and `hm1` (end time) to enable **TWAP** (Time-Weighted
Average Price) execution.  The server slices the order across the given window.

You can also supply `target_date` to schedule on a future business day.

In [ ]:
# Buy Hyundai Motor (A005380) at 10 % weight.
# Execution is spread from 09:00 to 12:20 on the next trading day.
result = trader.order(
    prod_id="A001500",
    weight=0.10,
    ls="L",
    hm0="09:00",
    hm1="12:20",
    target_date="2026-03-17",  # must be a business day
)
print(result)

## 7. Limit Order

Pass `limit_prc` to peg the order to a specific price.  
The server will not execute above (buy) or below (sell) this price.

In [ ]:
# Buy Samsung Electronics at a limit price of 55,000 KRW.
result = trader.order(
    prod_id="A005930",
    weight=0.05,
    ls="L",
    limit_prc=55_000,
)
print(result)

## 8. Stop Order

Pass `stop_prc` to set a stop-loss or stop-breakout trigger.  
The order activates only when the market price crosses `stop_prc`.

In [ ]:
# Sell Samsung Electronics if the price drops to 50,000 KRW (stop-loss).
result = trader.order(
    prod_id="A005930",
    weight=0.05,
    ls="S",
    stop_prc=50_000,
)
print(result)

## 9. Limit + Stop (Bracket Order)

Combine both parameters for a bracket: the order only activates after
`stop_prc` is touched and is then capped by `limit_prc`.

In [ ]:
# Enter a long position in KOSPI ETF (A069500) only if price breaks above
# 32,000, and cap the buy price at 32,500.
result = trader.order(
    prod_id="A069500",
    weight=0.08,
    ls="L",
    limit_prc=32_500,
    stop_prc=32_000,
    hm0="09:00",
    hm1="11:30",
)
print(result)

## 10. 주문 취소

| 메서드 | 설명 |
|---|---|
| `cancel_order(ord_id)` | 특정 주문 ID 취소 |
| `cancel_all_pending()` | 미접수 주문 전체 취소 |
| `cancel_pending_by_stock(prod_id)` | 특정 종목의 미접수 주문만 취소 |

> `ord_id`는 `get_orders()` 응답의 `"Id"` 필드 값입니다.  
> `cancel_all_pending()` / `cancel_pending_by_stock()`는 `connect_realtime()` 이후에 사용 가능합니다.

In [ ]:
# 특정 주문 ID 취소 — get_orders()의 "Id" 필드 값을 사용
cancel_result = trader.cancel_order(ord_id=1219617)
print(cancel_result)

In [ ]:
# 미접수 주문 전체 취소 (connect_realtime() 이후 사용 가능)
# trader.connect_realtime()
# time.sleep(3)
# cancelled = trader.cancel_all_pending()
# print(f"취소된 주문 수: {cancelled}")

In [ ]:
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")

# Portfolio/Orders — 당일 주문 목록 조회
res = api.get("Portfolio/Orders", params={"pfId": PF_ID, "d": today})
print(res.status_code)
# res.json()  # 전체 응답 페이로드

## 11. Raw API Access

`TimefolioAPIClient` exposes `get()` and `post()` helpers for any endpoint
not yet wrapped by `TimefolioTrader`.

In [ ]:
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")

# Example: raw GET to fetch portfolio details
res = api.get("Portfolio/Summary", params={"pfId": PF_ID, "d": today})
print(res.status_code)
# res.json()  # full response payload

---

## Parameter Reference

| Parameter | Type | Default | Description |
|---|---|---|---|
| `prod_id` | `str` | — | Ticker symbol, e.g. `"A005930"` |
| `weight` | `float` | — | Portfolio weight, `0.05` = 5 % |
| `ls` | `str` | `"L"` | `"L"` Long/Buy · `"S"` Short/Sell |
| `ex` | `str` | `"E"` | Execution algorithm type |
| `limit_idx` | `int` | `5` | Order-book depth aggressiveness (1–10) |
| `limit_prc` | `float\|None` | `None` | Exact limit price; `None` = algo/market |
| `stop_prc` | `float\|None` | `None` | Stop trigger price; `None` = disabled |
| `hm0` | `str\|None` | `None` | Start time `"HH:MM"`; `None` = immediate |
| `hm1` | `str\|None` | `None` | End time `"HH:MM"` for TWAP window |
| `target_date` | `str\|None` | `None` | Business date `"YYYY-MM-DD"`; `None` = today |

---

## 12. TimefolioCollector — 리더보드 수집

`TimefolioCollector`는 대회 리더보드와 참가자 포트폴리오를 수집하는 고수준 클라이언트입니다.  
로그인된 `api` 인스턴스를 그대로 전달하면 됩니다.

| 메서드 | 설명 |
|---|---|
| `get_leaderboard(date)` | 전체 참가자 순위 리스트 반환 |
| `get_top_leaders(date, top_n)` | 상위 N명 잘라서 반환 |
| `get_portfolio_detail(pf_id, date)` | 특정 참가자 포트폴리오 상세 |
| `get_holdings(pf_id, date)` | 보유종목 리스트 (`prfts` 키) |
| `collect_all_holdings(date, top_n, delay)` | 상위 N명 전체 보유종목 DataFrame |
| `get_violations(pf_id, date)` | 투자제한 위반 여부 |
| `save_snapshot(df, output_dir)` | CSV로 저장 |

In [5]:
import pandas as pd
from timefolio import TimefolioCollector

# api는 위 섹션 1에서 이미 로그인된 인스턴스
collector = TimefolioCollector(api_client=api, contest_id=86)
print(f"Collector ready (contest_id={collector.contest_id})")

Collector ready (contest_id=86)


### 전체 참가자 리더보드 조회

`get_leaderboard()`는 해당 날짜의 전체 참가자 목록을 반환합니다.  
응답을 DataFrame으로 변환하여 상위 10명을 확인합니다.

In [6]:
# 전체 리더보드 조회 (오늘 날짜 기준)
leaderboard = collector.get_leaderboard()
print(f"참가자 수: {len(leaderboard)}")

# 응답 구조 확인 (첫 번째 항목 키 출력)
if leaderboard:
    print("응답 키:", list(leaderboard[0].keys()))

# DataFrame 변환 후 상위 10명 출력
lb_df = pd.DataFrame(leaderboard)
lb_df.head(10)

참가자 수: 2085
응답 키: ['Id', 'userId', 'initD', 'lastD', 'initAmt', 'ctstId', 'currNav', 'currCh', 'ctstNm', 'ctstNmDisp', 'ctstInitD', 'strg', 'ty', 'userNick', 'active', 'rank', 'rankCh', 'excRank', 'afterReset', 'applyIntern', 'employed', 'nConn', 'retentionT', 'pfLabel', 'cmpl', 'stat', 'memo', 'memoHist']


,Id,userId,initD,lastD,initAmt,ctstId,currNav,currCh,ctstNm,ctstNmDisp,...,afterReset,applyIntern,employed,nConn,retentionT,pfLabel,cmpl,stat,memo,memoHist
0,20946,869292700,2026-04-03,2026-05-29,1000000000,86,None,None,RFM 11회 대회,None,...,True,None,None,None,None,None,"{'asr': 99.24363602951627, 'currWto': 57.6, 'a...","{'tfScAraw': [85.18, 62.98, 99.75, 98.79], 'tf...",None,[]
1,20061,865029638,2026-04-01,2026-05-29,1000000000,86,None,None,RFM 11회 대회,None,...,NaN,None,None,None,None,None,"{'asr': 91.26148647337905, 'currWto': 112.05, ...","{'tfScAraw': [63.47, 57.84, 99.96, 99.63], 'tf...",None,[]
2,20388,1760884460,2026-04-01,2026-05-29,1000000000,86,None,None,RFM 11회 대회,None,...,NaN,None,None,None,None,None,"{'asr': 99.31243482479312, 'currWto': 0, 'avgW...","{'tfScAraw': [72.76, 49.31, 32.74, 19.78], 'tf...",None,[]
3,18832,72482369,2026-04-01,2026-05-29,1000000000,86,None,None,RFM 11회 대회,None,...,NaN,None,None,None,None,None,"{'asr': 89.71091346795933, 'currWto': 41.5, 'a...","{'tfScAraw': [77.2, 46.47, 97.35, 100], 'tfScA...",None,[]
4,20506,1334594992,2026-04-01,2026-05-29,1000000000,86,None,None,RFM 11회 대회,None,...,NaN,None,None,None,None,None,"{'asr': 96.2619980910204, 'currWto': 5, 'avgWt...","{'tfScAraw': [79.09, 58.15, 54.61, 35.43], 'tf...",None,[]
5,18889,245641888,2026-04-01,2026-05-29,1000000000,86,None,None,RFM 11회 대회,None,...,NaN,None,None,None,None,None,"{'asr': 100.0104320174309, 'currWto': 51.75, '...","{'tfScAraw': [89.95, 65.99, 100, 99.93], 'tfSc...",None,[]
6,20632,1762430121,2026-04-01,2026-05-29,1000000000,86,None,None,RFM 11회 대회,None,...,NaN,None,None,None,None,None,"{'asr': 94.49506007004538, 'currWto': 7.8, 'av...","{'tfScAraw': [71.2, 50.98, 53, 24.97], 'tfScA'...",None,[]
7,19721,1636958072,2026-04-01,2026-05-29,1000000000,86,None,None,RFM 11회 대회,None,...,NaN,None,None,None,None,None,"{'asr': 89.49561769369988, 'currWto': 88, 'avg...","{'tfScAraw': [74.77, 40.23, 99.02, 73.11], 'tf...",None,[]
8,20405,1608610510,2026-04-01,2026-05-29,1000000000,86,None,None,RFM 11회 대회,None,...,NaN,None,None,None,None,None,"{'asr': 87.01094769436696, 'currWto': 58.25, '...","{'tfScAraw': [82.61, 39.55, 79.58, 78.58], 'tf...",None,[]
9,21371,1919143917,2026-04-07,2026-05-29,1000000000,86,None,None,RFM 11회 대회,None,...,True,None,None,None,None,None,"{'asr': 98.88309755178912, 'currWto': 66.45, '...","{'tfScAraw': [79.68, 54.22, 99.2, 91.68], 'tfS...",None,[]


In [7]:
# 상위 20명만 조회
top20 = collector.get_top_leaders(top_n=20)
print(f"상위 20명: {len(top20)}명")
for p in top20[:5]:
    print(p)

상위 20명: 20명
{'Id': 20946, 'userId': 869292700, 'initD': '2026-04-03', 'lastD': '2026-05-29', 'initAmt': 1000000000, 'ctstId': 86, 'currNav': None, 'currCh': None, 'ctstNm': 'RFM 11회 대회', 'ctstNmDisp': None, 'ctstInitD': '2026-04-01', 'strg': 'LongOnly', 'ty': 'Contest', 'userNick': '나카노 요츠바', 'active': False, 'rank': 1, 'rankCh': None, 'excRank': None, 'afterReset': True, 'applyIntern': None, 'employed': None, 'nConn': None, 'retentionT': None, 'pfLabel': None, 'cmpl': {'asr': 99.24363602951627, 'currWto': 57.6, 'avgWto': 68.03, 'wtoViols': 0, 'expWtoViols': 0}, 'stat': {'tfScAraw': [85.18, 62.98, 99.75, 98.79], 'tfScA': 82.86285714285714, 'tfScA0': 85.18, 'tfScA1': 62.98, 'tfScA2': 99.75, 'tfScA3': 98.79, 'days': 16, 'rt': 48.4921, 'rtStd': 43.0452, 'sharpe': 871.9044, 'mdd': 2.9, 'pfStd': 43.2673, 'rtBeta': 0.5984119668604805, 'rtIf100': 48.561, 'maxPfAsOfDiff': 0, 'topPrftNm': '코스텍시스', 'topPrftWei': 14.82, 'topNwei': 37.02, 'liqDays': 0.04, 'todStkWei': 100, 'avgPosCnt': 11.2, 'todP

---

## 13. 포트폴리오 상세 조회

`get_portfolio_detail()`로 특정 참가자의 전체 데이터를,  
`get_holdings()`로 보유종목 리스트만 추출합니다.

In [8]:
# 1위 참가자의 포트폴리오 상세 조회
# PfList 응답은 pfId가 아닌 Id(대문자) 키를 사용
if top20:
    first = top20[0]
    pf_id = first.get("Id") or first.get("pfId")
    nick  = first.get("userNick", str(pf_id))
    rank  = first.get("rank", 1)
    print(f"[{rank}위] {nick} (pfId={pf_id})")

    detail = collector.get_portfolio_detail(pf_id=pf_id)
    if detail:
        print("응답 최상위 키:", list(detail.keys()))
        print("\n--- pos (기본 정보) ---")
        print(detail.get("pos"))
    else:
        print("데이터 없음")

[1위] 나카노 요츠바 (pfId=20946)
응답 최상위 키: ['pos', 'prfts', 'daily', 'intra']

--- pos (기본 정보) ---
[]


In [9]:
# 보유종목 리스트만 추출
if top20:
    holdings = collector.get_holdings(pf_id=pf_id)
    print(f"보유종목 수: {len(holdings)}")
    pd.DataFrame(holdings) if holdings else print("보유종목 없음")

보유종목 수: 10


---

## 14. 전체 수집 — collect_all_holdings

상위 N명의 보유종목을 한 번에 수집합니다.  
각 요청 사이에 `delay`(초) 만큼 대기하여 서버 부담을 줄입니다.

In [ ]:
# 상위 20명 보유종목 전체 수집
# delay=0.5 → 참가자 1명당 0.5초 대기
df = collector.collect_all_holdings(top_n=20, delay=0.5)
print(f"수집 완료: {len(df)}행 × {len(df.columns)}열")
df.head(10)

In [ ]:
# CSV로 저장 (runs/{date}/holdings_top{n}.csv)
if not df.empty:
    path = collector.save_snapshot(df, output_dir="runs")
    print(f"저장 완료: {path}")

---

## 15. 간단 분석 — 상위 20명이 가장 많이 보유한 종목

`groupby`로 종목별 보유자 수를 집계하여 Top 15 종목을 확인합니다.

In [ ]:
if not df.empty:
    # 실제 컬럼명 확인 (API 응답에 따라 달라질 수 있음)
    print("DataFrame 컬럼:", df.columns.tolist())

    # prodId / prodNm 컬럼명 자동 탐지
    id_col  = next((c for c in df.columns if "prodId" in c or "prodid" in c.lower()), None)
    nm_col  = next((c for c in df.columns if "prodNm" in c or "prodnm" in c.lower()), None)
    pf_col  = "participant_pfid"

    if id_col and pf_col in df.columns:
        top_stocks = (
            df.groupby([id_col, nm_col] if nm_col else [id_col])[pf_col]
            .nunique()
            .rename("보유자_수")
            .sort_values(ascending=False)
            .head(15)
            .reset_index()
        )
        print("\n[ 상위 20명이 가장 많이 보유한 종목 Top 15 ]")
        print(top_stocks.to_string(index=False))
    else:
        print("prodId 또는 participant_pfid 컬럼을 찾을 수 없습니다.")
        print("현재 컬럼:", df.columns.tolist())
else:
    print("수집된 데이터가 없습니다.")